In [1]:
#------------------------------------------------ Begin_Librairie ----------------------------------------
from bs4 import BeautifulSoup
import requests
import datetime
import pandas as pd
import re
import os
from time import sleep

In [2]:
#------------------------------------------------ Begin_ fileName ----------------------------------------
regulatorName = 'SM BCSM'

print(f"Running {regulatorName} Web Scraping Tool v.1.4")
now = datetime.datetime.now()
filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":", ".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"

#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment
os.chdir(scriptfolder)
tempfolder = os.path.join(scriptfolder, 'tempfolder')

if os.path.exists(tempfolder):
    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))
else:
    os.mkdir(tempfolder)

Running SM BCSM Web Scraping Tool v.1.4


In [3]:
#------------------------------------------------ Begin_Variable ----------------------------------------

sqldict = {
    'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [],
    'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [],
    'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [],
    'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [],
    'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [],
    'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [],
    'RegCtry': [], 'RegCode': [], 'ListCode': [], 'ListLanguage': [],
    'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [],
    'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
    'Address_1 - Mother company': [], 'Address_2 -  Mother company': [],
    'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [],
    'Phone - Mother company': [], 'Check': []
}

processdate = now.strftime('%Y-%m-%d')

# List 1 scrapes two sub-pages (SM firms + foreign firms), both under ListCode '1'
list1_urls = [
    'https://www.bcsm.sm/registro-soggetti-autorizzati-imprese-finanziarie-sammarinesi',
    'https://www.bcsm.sm/imprese-finanziarie-estere',
]

regdict = {
    # regulatorName + ' 1': list1_urls,
    # regulatorName + ' 3': 'https://www.bcsm.sm/intermediari-assicurativi-e-riassicurativi',
    # regulatorName + ' 5': 'https://www.bcsm.sm/registro-dei-prestatori-di-servizi-di-pagamento',
    # regulatorName + ' 6': 'https://www.bcsm.sm/registri/albo-dei-trustee-professionali-della-repubblica-di-san-marino',
    regulatorName + ' 7': 'https://www.bcsm.sm/registro-imprese-capogruppo',
    # regulatorName + ' 8': 'https://www.bcsm.sm/elenco-fondi-approvati',
}

Typology = {
    regulatorName + ' 1': 'Register of authorized entities',
    regulatorName + ' 3': 'Register of insurance and reinsurance intermediaries',
    regulatorName + ' 5': 'Register of payment service providers',
    regulatorName + ' 6': 'Register of Professional Trustees in the Republic Of San Marino',
    regulatorName + ' 7': 'Register of holding companies',
    regulatorName + ' 8': 'List of Approved Funds (EFA)',
}

In [4]:
#------------------------------------------------ Begin_Fouction ----------------------------------------
def bourange_same_length_array(sqldict):
    maxlen = len(sqldict['ListProcessDate'])
    for key in sqldict:
        if len(sqldict[key]) != maxlen:
            sqldict[key] = sqldict[key] + [''] * (maxlen - len(sqldict[key]))
    return sqldict


def fetch_soup(url):
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    return BeautifulSoup(resp.content, 'html.parser')


def parse_field(text, label):
    """Extract the value after 'label:' from a text block."""
    pattern = re.compile(re.escape(label) + r'\s*:\s*(.+)', re.IGNORECASE)
    m = pattern.search(text)
    return m.group(1).strip() if m else ''


def _normalise_label(label):
    import unicodedata

    text = unicodedata.normalize('NFKD', str(label))
    text = ''.join(ch for ch in text if not unicodedata.combining(ch))
    text = re.sub(r'\s+', ' ', text).strip().rstrip(':').lower()
    return text


def country_to_iso(country_name):
    """Normalize BCSM country names to ISO alpha-2 codes."""
    import unicodedata

    if country_name is None:
        return ''

    cleaned = unicodedata.normalize('NFKD', str(country_name))
    cleaned = ''.join(ch for ch in cleaned if not unicodedata.combining(ch))
    cleaned = cleaned.replace('’', "'").replace('`', "'").replace('´', "'")
    cleaned = re.sub(r'\s+', ' ', cleaned).strip().upper()

    if re.fullmatch(r'[A-Z]{2}', cleaned):
        return cleaned

    lookup_key = re.sub(r"[.'’`´]", '', cleaned)
    lookup_key = re.sub(r'\s+', ' ', lookup_key).strip()

    country_map = {
        'LUSSEMBURGO': 'LU',
        'ITALIA': 'IT',
        'GERMANIA': 'DE',
        'SPAGNA': 'ES',
        'FRANCIA': 'FR',
        'REGNO UNITO': 'GB',
        'BELGIO': 'BE',
        'PORTOGALLO': 'PT',
        'MALTA': 'MT',
        'IRLANDA': 'IE',
        'LIECHTENSTEIN': 'LI',
        'SVIZZERA': 'CH',
        'REPUBBLICA DI SAN MARINO': 'SM',
        'SAN MARINO': 'SM',
    }
    return country_map.get(lookup_key, cleaned)


def extract_label_value(container, label):
    """Extract text after a matching <b>label</b> up to the next <br> or label."""
    if not container:
        return ''

    from bs4 import NavigableString, Tag

    def node_in_container(node):
        return node is container or container in getattr(node, 'parents', [])

    target = _normalise_label(label)
    for bold in container.find_all('b'):
        if _normalise_label(bold.get_text(' ', strip=True)) != target:
            continue

        parts = []
        for node in bold.next_elements:
            if not node_in_container(node):
                break
            if isinstance(node, Tag) and node.name in ('br', 'b'):
                break
            if not isinstance(node, NavigableString):
                continue
            if bold in getattr(node, 'parents', []):
                continue

            text = str(node).strip()
            if text:
                parts.append(text)

        value = re.sub(r'\s+', ' ', ' '.join(parts)).strip()
        return re.sub(r'^:\s*', '', value).strip()

    return ''


def extract_first_label_value(container, labels):
    """Return the first non-empty structured label value from a list of labels."""
    for label in labels:
        value = extract_label_value(container, label)
        if value:
            return value
    return ''


def parse_address_section(section):
    return {
        'Indirizzo': extract_first_label_value(section, ['Indirizzo', 'Strada, Via, Piazza e Numero civico']),
        'Località': extract_first_label_value(section, ['Località', 'Città/Località']),
        'CAP': extract_label_value(section, 'CAP'),
        'Castello': extract_label_value(section, 'Castello'),
        'Stato': extract_label_value(section, 'Stato'),
    }


def build_detail_url(href):
    if not href:
        return ''
    return 'https://www.bcsm.sm' + href if href.startswith('/') else href


def find_section_by_heading(soup, heading_names):
    """Collect the HTML between a matching heading and the next peer heading/HR."""
    if not soup:
        return None

    from bs4 import NavigableString, Tag

    normalised_names = [_normalise_label(name) for name in heading_names]
    for heading in soup.find_all(re.compile(r'^h[1-6]$')):
        heading_text = _normalise_label(heading.get_text(' ', strip=True))
        if not any(name in heading_text for name in normalised_names):
            continue

        level = int(heading.name[1])
        container = BeautifulSoup('', 'html.parser')
        sibling = heading.next_sibling
        while sibling:
            if isinstance(sibling, Tag):
                if sibling.name == 'hr':
                    break
                if re.fullmatch(r'h[1-6]', sibling.name or '') and int(sibling.name[1]) <= level:
                    break
                container.append(sibling.__copy__())
            else:
                container.append(NavigableString(str(sibling)))
            sibling = sibling.next_sibling

        if container.get_text(' ', strip=True):
            return container

    return None


def parse_detail_address(detail_soup):
    empty_address = {'Address_1': '', 'Address_2': '', 'City': '', 'Zip': '', 'Cntry': ''}
    if not detail_soup:
        return empty_address

    sede_legale_section = (
        detail_soup.select_one('.altri-dati-societari .sede-legale')
        or detail_soup.select_one('.sede-legale')
        or find_section_by_heading(detail_soup, ['Sede Legale', 'Sede principale'])
    )
    sede_amministrativa_section = (
        detail_soup.select_one('.altri-dati-societari .sede-amministrativa')
        or detail_soup.select_one('.sede-amministrativa')
        or find_section_by_heading(detail_soup, ['Sede Amministrativa'])
    )

    sede_legale = parse_address_section(sede_legale_section)
    if not any(sede_legale.values()):
        sede_legale = parse_address_section(detail_soup)
    sede_amministrativa = parse_address_section(sede_amministrativa_section)

    return {
        'Address_1': sede_legale['Indirizzo'],
        'Address_2': sede_amministrativa['Indirizzo'],
        'City': sede_legale['Località'],
        'Zip': sede_legale['CAP'],
        'Cntry': country_to_iso(sede_legale['Stato']),
    }


def parse_text_field(text, label, stop_labels=None):
    """Extract a label value from flattened listing text, stopping at known labels."""
    if not text:
        return ''

    if stop_labels is None:
        stop_labels = [
            'Numero iscrizione', 'N. iscrizione', 'Codice fondo', 'Denominazione',
            'Denominazione / Cognome e Nome', 'Forma giuridica', 'Sede legale',
            'C.O.E.', 'Codice Operatore Economico', 'Data autorizzazione',
            'Data iscrizione', 'Data cancellazione', 'Responsabile ufficio trustee (RUT)',
            'Vice RUT', 'Stato', 'Note', 'Sezione', 'Servizi di pagamento autorizzati',
            'Agenti', 'Tipologia', 'Numero iscrizione al Registro dei Soggetti Autorizzati',
        ]

    label_pattern = re.escape(label).replace(r'\ ', r'\s+')
    stop_patterns = [
        re.escape(stop_label).replace(r'\ ', r'\s+')
        for stop_label in stop_labels
        if _normalise_label(stop_label) != _normalise_label(label)
    ]
    stop_pattern = '|'.join(stop_patterns)
    pattern = re.compile(label_pattern + r'\s*:\s*(.*?)(?=\s+(?:' + stop_pattern + r')\s*:|$)', re.IGNORECASE)
    match = pattern.search(text)
    return re.sub(r'\s+', ' ', match.group(1)).strip() if match else ''


def parse_flat_address(address_text):
    parsed = {'Address_1': '', 'City': '', 'Zip': '', 'Cntry': ''}
    if not address_text:
        return parsed

    country_match = re.search(r'\(([^)]+)\)\s*$', address_text)
    if country_match:
        parsed['Cntry'] = country_to_iso(country_match.group(1))
        address_text = address_text[:country_match.start()].strip()

    if ' - ' in address_text:
        parsed['Address_1'], remainder = [part.strip() for part in address_text.split(' - ', 1)]
    else:
        parsed['Address_1'], remainder = address_text.strip(), ''

    parts = [part.strip() for part in remainder.split(',') if part.strip()]
    if parts and re.fullmatch(r'\d{4,6}', parts[0]):
        parsed['Zip'] = parts.pop(0)
    if parts:
        parsed['City'] = parts[0]

    return parsed


def regulation_type_from_status(status):
    status_lower = (status or '').lower()
    if 'inattivo' in status_lower or 'cancellato' in status_lower:
        return 'Inactive'
    if 'attivo' in status_lower:
        return 'Regulated'
    return 'Regulated'


def split_by_hr(soup):
    """Split page content into entity blocks delimited by <hr> tags.
    Returns list of BeautifulSoup Tag fragments (one per entity)."""
    from bs4 import NavigableString, Tag

    def build_block(block_elements):
        if not block_elements:
            return None

        container = BeautifulSoup('', 'html.parser')
        for el in block_elements:
            container.append(el.__copy__() if isinstance(el, Tag) else NavigableString(str(el)))

        text = container.get_text(' ', strip=True)
        if not text or len(text) <= 5:
            return None

        link = container.find('a')
        return {
            'text': text,
            'link_name': link.get_text(strip=True) if link else '',
            'link_href': link.get('href', '') if link else '',
            'soup': container,
        }

    hrs = soup.find_all('hr')
    blocks = []

    if hrs:
        first_hr = hrs[0]
        block_elements = []
        sibling = first_hr.previous_sibling
        while sibling:
            if isinstance(sibling, Tag) and sibling.name == 'hr':
                break
            block_elements.insert(0, sibling)
            sibling = sibling.previous_sibling

        first_block = build_block(block_elements)
        listing_label_pattern = r'(Denominazione|Denominazione gruppo|Denominazione / Cognome e Nome|Numero iscrizione|N. iscrizione|Codice fondo)\s*:'
        if first_block and re.search(listing_label_pattern, first_block['text'], re.IGNORECASE):
            blocks.append(first_block)

    for hr in hrs:
        block_elements = []
        sibling = hr.next_sibling
        while sibling:
            if isinstance(sibling, Tag) and sibling.name == 'hr':
                break
            block_elements.append(sibling)
            sibling = sibling.next_sibling

        block = build_block(block_elements)
        if block:
            blocks.append(block)

    return blocks


def append_common(sqldict, list_code, processdate, typology_name, reg_type='Regulated'):
    sqldict['RegCtry'].append('SM')
    sqldict['RegCode'].append('BCSM')
    sqldict['ListCode'].append(list_code)
    sqldict['ListProcessDate'].append(processdate)
    sqldict['ListName'].append(typology_name)
    sqldict['RegulationType'].append(reg_type)
    return sqldict

In [5]:
#------------------------------------------------ Begin_Main ----------------------------------------
for k, reg in enumerate(regdict):
    list_code = reg.split()[-1]
    typology_name = Typology[reg]
    urls = regdict[reg] if isinstance(regdict[reg], list) else [regdict[reg]]

    print(f"[INFO] : Working {k+1}/{len(regdict)} _({reg})_ ")

    for url in urls:
        print(f"  Fetching: {url}")
        soup = fetch_soup(url)
        blocks = split_by_hr(soup)
        print(f"  Found {len(blocks)} entity blocks")

        # ----------------------------------------------------------------
        # List 1: Register of authorized entities
        #   Follows each entity's detail-page link to extract rich info
        # ----------------------------------------------------------------
        if list_code == '1':
            is_foreign = 'estere' in url
            total = len([b for b in blocks if b['link_name']])
            entity_i = 0
            for b in blocks:
                name = b['link_name']
                if not name:
                    continue
                entity_i += 1

                href = b['link_href']
                if href.startswith('/'):
                    detail_url = 'https://www.bcsm.sm' + href
                else:
                    detail_url = href

                if entity_i % 10 == 1 or entity_i == total:
                    print(f"    Detail {entity_i}/{total}: {name}")

                try:
                    detail_soup = fetch_soup(detail_url)
                except Exception as e:
                    print(f"    [WARN] Could not fetch detail for {name}: {e}")
                    detail_soup = BeautifulSoup('', 'html.parser')

                sleep(0.5)

                detail_name = extract_label_value(detail_soup, 'Denominazione')
                n_iscrizione = (
                    extract_label_value(detail_soup, 'N. iscrizione')
                    or extract_label_value(detail_soup, 'Numero iscrizione')
                )
                data_iscrizione = extract_label_value(detail_soup, 'Data iscrizione')
                data_cancellazione = extract_label_value(detail_soup, 'Data cancellazione')
                forma_giuridica = extract_label_value(detail_soup, 'Forma giuridica')
                coe = extract_label_value(detail_soup, 'Codice Operatore Economico')
                website = extract_label_value(detail_soup, 'Sito internet')

                if data_cancellazione and data_cancellazione.strip() != '-':
                    reg_type = 'Cancelled'
                else:
                    reg_type = 'Regulated'

                sede_legale_section = (
                    detail_soup.select_one('.altri-dati-societari .sede-legale')
                    or detail_soup.select_one('.sede-legale')
                )
                sede_amministrativa_section = (
                    detail_soup.select_one('.altri-dati-societari .sede-amministrativa')
                    or detail_soup.select_one('.sede-amministrativa')
                )

                sede_legale = parse_address_section(sede_legale_section)
                sede_amministrativa = parse_address_section(sede_amministrativa_section)

                address_1 = sede_legale['Indirizzo']
                address_2 = sede_amministrativa['Indirizzo']
                city = sede_legale['Località']
                zipcode = sede_legale['CAP']
                cntry = sede_legale['Stato'] if is_foreign else 'SM'

                if coe:
                    internal_id_1 = coe
                    internal_id_1_type = 'Codice Operatore Economico'
                    internal_id_2 = n_iscrizione
                    internal_id_2_type = 'N. iscrizione (Registro Soggetti Autorizzati)' if n_iscrizione else ''
                else:
                    internal_id_1 = n_iscrizione
                    internal_id_1_type = 'N. iscrizione (Registro Soggetti Autorizzati)' if n_iscrizione else ''
                    internal_id_2 = ''
                    internal_id_2_type = ''

                sqldict['Name'].append(detail_name or name)
                sqldict['InternalID_1'].append(internal_id_1.split[' '][0])
                sqldict['InternalID_1_type'].append(internal_id_1_type)
                sqldict['InternalID_2'].append(internal_id_2)
                sqldict['InternalID_2_type'].append(internal_id_2_type)
                # sqldict['CoType'].append(forma_giuridica)
                sqldict['Address_1'].append(address_1)
                sqldict['Address_2'].append(address_2)
                sqldict['City'].append(city)
                sqldict['Zip'].append(zipcode)
                sqldict['Cntry'].append(country_to_iso(cntry))
                sqldict['Website'].append(website)
                sqldict['RegulationDate'].append(data_iscrizione)

                append_common(sqldict, list_code, processdate, typology_name, reg_type)

        # ----------------------------------------------------------------
        # List 3: Insurance and reinsurance intermediaries
        #   Follows each entity's detail link and parses the structured detail HTML
        # ----------------------------------------------------------------
        elif list_code == '3':
            def list3_find_heading(detail_soup, heading_id):
                if not detail_soup:
                    return None

                heading = detail_soup.find('h4', id=heading_id)
                if heading:
                    return heading

                target = _normalise_label(heading_id.replace('-', ' '))
                for heading in detail_soup.find_all('h4'):
                    if _normalise_label(heading.get_text(' ', strip=True)) == target:
                        return heading
                return None

            def list3_heading_paragraph(detail_soup, heading_id):
                from bs4 import Tag

                heading = list3_find_heading(detail_soup, heading_id)
                if not heading:
                    return None

                for element in heading.next_elements:
                    if not isinstance(element, Tag):
                        continue
                    if element.name in ('h4', 'hr'):
                        break
                    if element.name == 'p':
                        return element

                parent = heading.find_parent(class_='note') or heading.parent
                if parent:
                    for paragraph in parent.find_all('p'):
                        if paragraph.find_previous('h4') == heading:
                            return paragraph
                return None

            def list3_first_succursale_address(detail_soup):
                from bs4 import Tag

                heading = list3_find_heading(detail_soup, 'succursali')
                if not heading:
                    return ''

                table = None
                for element in heading.next_elements:
                    if not isinstance(element, Tag):
                        continue
                    if element.name in ('h4', 'hr'):
                        break
                    if element.name == 'table':
                        table = element
                        break

                if not table:
                    container = heading.find_parent('div') or heading
                    table = container.find('table')
                if not table:
                    return ''

                for row in table.find_all('tr'):
                    cells = row.find_all('td')
                    if cells:
                        return cells[0].get_text(' ', strip=True)
                return ''

            total = len([b for b in blocks if b['link_name']])
            entity_i = 0
            for b in blocks:
                name = b['link_name']
                if not name:
                    continue
                entity_i += 1

                detail_url = build_detail_url(b['link_href'])
                detail_soup = BeautifulSoup('', 'html.parser')
                if detail_url:
                    if entity_i % 10 == 0 or entity_i == total:
                        print(f"    Detail {entity_i}/{total}: {name}")
                    try:
                        detail_soup = fetch_soup(detail_url)
                    except Exception as e:
                        print(f"    [WARN] Could not fetch detail for {name}: {e}")
                    sleep(0.5)

                dati_generali = list3_heading_paragraph(detail_soup, 'dati-generali')
                sede_principale = list3_heading_paragraph(detail_soup, 'sede-principale')

                detail_name = extract_label_value(dati_generali, 'Nome ditta individuale o denominazione/ragione sociale')
                coe = extract_label_value(dati_generali, 'Codice Operatore Economico')
                license_type = extract_label_value(dati_generali, "Tipologia dell'attività di distribuzione")
                ruolo_professionale = extract_label_value(dati_generali, 'Ruolo professionale')

                n_iscrizione = extract_label_value(detail_soup, 'N. iscrizione')
                data_iscrizione = extract_label_value(detail_soup, 'Data iscrizione')
                sezione = extract_label_value(detail_soup, 'Sezione')

                sede = parse_address_section(sede_principale)
                address_2 = list3_first_succursale_address(detail_soup)

                sqldict['Name'].append(detail_name or name)
                sqldict['InternalID_1'].append(coe)
                sqldict['InternalID_1_type'].append('Codice Operatore Economico' if coe else '')
                sqldict['InternalID_2'].append(n_iscrizione)
                sqldict['InternalID_2_type'].append(
                    'N. iscrizione (Registro Intermediari Assicurativi e Riassicurativi)' if n_iscrizione else ''
                )
                # sqldict['CoType'].append(ruolo_professionale)
                # sqldict['License_Type'].append(license_type)
                sqldict['Address_1'].append(sede['Indirizzo'])
                sqldict['Address_2'].append(address_2)
                sqldict['City'].append(sede['Località'])
                sqldict['Zip'].append(sede['CAP'])
                sqldict['Cntry'].append(country_to_iso(sede['Stato']))
                sqldict['RegulationDate'].append(data_iscrizione)
                sqldict['Typology'].append(sezione)

                append_common(sqldict, list_code, processdate, typology_name, 'Regulated')

        # ----------------------------------------------------------------
        # List 5: Payment service providers
        #   Follows detail links for COE/address; preserves PSP registration
        # ----------------------------------------------------------------
        elif list_code == '5':
            total = len([b for b in blocks if b['link_name']])
            entity_i = 0
            for b in blocks:
                txt = b['text']
                name = b['link_name']
                if not name:
                    continue
                entity_i += 1

                detail_url = build_detail_url(b['link_href'])
                detail_soup = BeautifulSoup('', 'html.parser')
                if detail_url and 'bcsm.sm' in detail_url:
                    if entity_i % 10 == 0 or entity_i == total:
                        print(f"    Detail {entity_i}/{total}: {name}")
                    try:
                        detail_soup = fetch_soup(detail_url)
                    except Exception as e:
                        print(f"    [WARN] Could not fetch detail for {name}: {e}")
                    sleep(0.5)

                detail_name = extract_label_value(detail_soup, 'Denominazione')
                n_iscrizione = (
                    parse_text_field(txt, 'Numero iscrizione')
                    or extract_first_label_value(detail_soup, ['Numero iscrizione', 'N. iscrizione'])
                )
                data_iscrizione = parse_text_field(txt, 'Data iscrizione') or extract_label_value(detail_soup, 'Data iscrizione')
                license_type = parse_text_field(txt, 'Servizi di pagamento autorizzati')
                stato = extract_label_value(detail_soup, 'Stato') or parse_text_field(txt, 'Stato')
                coe = extract_first_label_value(detail_soup, ['Codice Operatore Economico', 'C.O.E.'])
                website = extract_label_value(detail_soup, 'Sito internet')
                forma_giuridica = extract_label_value(detail_soup, 'Forma giuridica')
                address = parse_detail_address(detail_soup)

                if coe:
                    internal_id_1 = coe
                    internal_id_1_type = 'Codice Operatore Economico'
                    internal_id_2 = n_iscrizione
                    internal_id_2_type = 'Numero iscrizione (Registro dei Prestatori di Servizi di Pagamento)' if n_iscrizione else ''
                else:
                    internal_id_1 = n_iscrizione
                    internal_id_1_type = 'Numero iscrizione (Registro dei Prestatori di Servizi di Pagamento)' if n_iscrizione else ''
                    internal_id_2 = ''
                    internal_id_2_type = ''

                sqldict['Name'].append(detail_name or name)
                sqldict['InternalID_1'].append(internal_id_1)
                sqldict['InternalID_1_type'].append(internal_id_1_type)
                sqldict['InternalID_2'].append(internal_id_2)
                sqldict['InternalID_2_type'].append(internal_id_2_type)
                # sqldict['CoType'].append(forma_giuridica)
                # sqldict['License_Type'].append(license_type)
                sqldict['Address_1'].append(address['Address_1'])
                sqldict['Address_2'].append(address['Address_2'])
                sqldict['City'].append(address['City'])
                sqldict['Zip'].append(address['Zip'])
                sqldict['Cntry'].append(address['Cntry'])
                sqldict['Website'].append(website)
                sqldict['RegulationDate'].append(data_iscrizione)

                append_common(sqldict, list_code, processdate, typology_name, regulation_type_from_status(stato))

        # ----------------------------------------------------------------
        # List 6: Professional Trustees
        #   Follows detail links when present; otherwise parses listing labels
        # ----------------------------------------------------------------
        elif list_code == '6':
            total = len(blocks)
            entity_i = 0
            for b in blocks:
                txt = b['text']
                listing_name = parse_text_field(txt, 'Denominazione / Cognome e Nome') or b['link_name']
                if not listing_name:
                    continue
                entity_i += 1

                detail_url = build_detail_url(b['link_href'])
                detail_soup = BeautifulSoup('', 'html.parser')
                if detail_url:
                    if entity_i % 10 == 0 or entity_i == total:
                        print(f"    Detail {entity_i}/{total}: {listing_name}")
                    try:
                        detail_soup = fetch_soup(detail_url)
                    except Exception as e:
                        print(f"    [WARN] Could not fetch detail for {listing_name}: {e}")
                    sleep(0.5)
                elif entity_i % 10 == 0 or entity_i == total:
                    print(f"    Listing fallback {entity_i}/{total}: {listing_name}")

                detail_name = extract_first_label_value(
                    detail_soup,
                    ['Denominazione', 'Denominazione / Cognome e Nome'],
                )
                n_iscrizione = (
                    extract_first_label_value(detail_soup, ['Numero iscrizione', 'N. iscrizione'])
                    or parse_text_field(txt, 'Numero iscrizione')
                )
                coe = (
                    extract_first_label_value(detail_soup, ['Codice Operatore Economico', 'C.O.E.'])
                    or parse_text_field(txt, 'C.O.E.')
                    or parse_text_field(txt, 'Codice Operatore Economico')
                )
                forma_giuridica = extract_label_value(detail_soup, 'Forma giuridica') or parse_text_field(txt, 'Forma giuridica')
                address = parse_detail_address(detail_soup)
                if not address['Address_1']:
                    flat_address = parse_flat_address(parse_text_field(txt, 'Sede legale'))
                    address = {
                        'Address_1': flat_address['Address_1'],
                        'Address_2': '',
                        'City': flat_address['City'],
                        'Zip': flat_address['Zip'],
                        'Cntry': flat_address['Cntry'],
                    }
                data_autorizzazione = extract_label_value(detail_soup, 'Data autorizzazione') or parse_text_field(txt, 'Data autorizzazione')
                stato = extract_label_value(detail_soup, 'Stato') or parse_text_field(txt, 'Stato')
                website = extract_label_value(detail_soup, 'Sito internet')

                if coe:
                    internal_id_1 = coe
                    internal_id_1_type = 'Codice Operatore Economico'
                    internal_id_2 = n_iscrizione
                    internal_id_2_type = 'Numero iscrizione (Albo dei Trustee Professionali)' if n_iscrizione else ''
                else:
                    internal_id_1 = n_iscrizione
                    internal_id_1_type = 'Numero iscrizione (Albo dei Trustee Professionali)' if n_iscrizione else ''
                    internal_id_2 = ''
                    internal_id_2_type = ''

                sqldict['Name'].append(detail_name or listing_name)
                sqldict['InternalID_1'].append(internal_id_1)
                sqldict['InternalID_1_type'].append(internal_id_1_type)
                sqldict['InternalID_2'].append(internal_id_2)
                sqldict['InternalID_2_type'].append(internal_id_2_type)
                # sqldict['CoType'].append(forma_giuridica)
                sqldict['Address_1'].append(address['Address_1'])
                sqldict['Address_2'].append(address['Address_2'])
                sqldict['City'].append(address['City'])
                sqldict['Zip'].append(address['Zip'])
                sqldict['Cntry'].append(country_to_iso(address['Cntry'] or 'SM'))
                sqldict['Website'].append(website)
                sqldict['RegulationDate'].append(data_autorizzazione)

                append_common(sqldict, list_code, processdate, typology_name, regulation_type_from_status(stato))

        # ----------------------------------------------------------------
        # List 7: Holding companies
        #   Follows group detail; second-hop to authorized entity if address absent
        # ----------------------------------------------------------------
        elif list_code == '7':
            total = len([b for b in blocks if b['link_name']])
            entity_i = 0
            for b in blocks:
                txt = b['text']
                name = b['link_name']
                if not name:
                    continue
                entity_i += 1

                detail_url = build_detail_url(b['link_href'])
                detail_soup = BeautifulSoup('', 'html.parser')
                if detail_url:
                    if entity_i % 10 == 0 or entity_i == total:
                        print(f"    Detail {entity_i}/{total}: {name}")
                    try:
                        detail_soup = fetch_soup(detail_url)
                    except Exception as e:
                        print(f"    [WARN] Could not fetch detail for {name}: {e}")
                    sleep(0.5)

                detail_name = extract_label_value(detail_soup, 'Denominazione') or name
                n_iscrizione = (
                    extract_first_label_value(detail_soup, ['Numero iscrizione', 'N. iscrizione'])
                    or parse_text_field(txt, 'Numero iscrizione')
                )
                data_iscrizione = extract_label_value(detail_soup, 'Data iscrizione') or parse_text_field(txt, 'Data iscrizione')
                tipologia = extract_label_value(detail_soup, 'Tipologia')
                stato = extract_label_value(detail_soup, 'Stato') or parse_text_field(txt, 'Stato')
                coe = extract_first_label_value(detail_soup, ['Codice Operatore Economico', 'C.O.E.'])
                website = extract_label_value(detail_soup, 'Sito internet')
                forma_giuridica = extract_label_value(detail_soup, 'Forma giuridica')
                address = parse_detail_address(detail_soup)

                if not address['Address_1']:
                    dati_generali = find_section_by_heading(detail_soup, ['Dati generali'])
                    auth_link = (dati_generali or detail_soup).find(
                        'a', href=re.compile(r'(registro-soggetti-autorizzati|imprese-finanziarie-estere)')
                    )
                    auth_url = build_detail_url(auth_link.get('href', '')) if auth_link else ''
                    if auth_url:
                        try:
                            auth_soup = fetch_soup(auth_url)
                            sleep(0.5)
                            address = parse_detail_address(auth_soup)
                            coe = coe or extract_first_label_value(auth_soup, ['Codice Operatore Economico', 'C.O.E.'])
                            website = website or extract_label_value(auth_soup, 'Sito internet')
                            forma_giuridica = forma_giuridica or extract_label_value(auth_soup, 'Forma giuridica')
                            detail_name = extract_label_value(auth_soup, 'Denominazione') or detail_name
                        except Exception as e:
                            print(f"    [WARN] Could not fetch linked entity for {name}: {e}")

                if coe:
                    internal_id_1 = coe
                    internal_id_1_type = 'Codice Operatore Economico'
                    internal_id_2 = n_iscrizione
                    internal_id_2_type = 'Numero iscrizione (Registro imprese capogruppo)' if n_iscrizione else ''
                else:
                    internal_id_1 = n_iscrizione
                    internal_id_1_type = 'Numero iscrizione (Registro imprese capogruppo)' if n_iscrizione else ''
                    internal_id_2 = ''
                    internal_id_2_type = ''

                sqldict['Name'].append(detail_name or name)
                sqldict['InternalID_1'].append(internal_id_1)
                sqldict['InternalID_1_type'].append(internal_id_1_type)
                sqldict['InternalID_2'].append(internal_id_2)
                sqldict['InternalID_2_type'].append(internal_id_2_type)
                # sqldict['CoType'].append(forma_giuridica)
                sqldict['Address_1'].append(address['Address_1'])
                sqldict['Address_2'].append(address['Address_2'])
                sqldict['City'].append(address['City'])
                sqldict['Zip'].append(address['Zip'])
                sqldict['Cntry'].append(country_to_iso(address['Cntry']))
                sqldict['Website'].append(website)
                sqldict['RegulationDate'].append(data_iscrizione)
                sqldict['Typology'].append(tipologia)

                append_common(sqldict, list_code, processdate, typology_name, regulation_type_from_status(stato))

        # ----------------------------------------------------------------
        # List 8: Approved Funds (EFA)
        #   Follows fund detail links for fund code and available structured data
        # ----------------------------------------------------------------
        elif list_code == '8':
            total = len([b for b in blocks if b['link_name']])
            entity_i = 0
            for b in blocks:
                txt = b['text']
                name = b['link_name']
                if not name:
                    continue
                entity_i += 1

                detail_url = build_detail_url(b['link_href'])
                detail_soup = BeautifulSoup('', 'html.parser')
                if detail_url:
                    if entity_i % 10 == 0 or entity_i == total:
                        print(f"    Detail {entity_i}/{total}: {name}")
                    try:
                        detail_soup = fetch_soup(detail_url)
                    except Exception as e:
                        print(f"    [WARN] Could not fetch detail for {name}: {e}")
                    sleep(0.5)

                detail_name = extract_label_value(detail_soup, 'Denominazione')
                codice_fondo = (
                    extract_first_label_value(detail_soup, ['Codice fondo', 'N. iscrizione', 'Numero iscrizione'])
                    or parse_text_field(txt, 'Codice fondo')
                )
                data_iscrizione = extract_label_value(detail_soup, 'Data iscrizione') or parse_text_field(txt, 'Data iscrizione')
                stato = extract_label_value(detail_soup, 'Stato') or parse_text_field(txt, 'Stato')
                coe = extract_first_label_value(detail_soup, ['Codice Operatore Economico', 'C.O.E.'])
                website = extract_label_value(detail_soup, 'Sito internet')
                forma_giuridica = extract_label_value(detail_soup, 'Forma giuridica')
                address = parse_detail_address(detail_soup)

                if coe:
                    internal_id_1 = coe
                    internal_id_1_type = 'Codice Operatore Economico'
                    internal_id_2 = codice_fondo
                    internal_id_2_type = 'Codice fondo' if codice_fondo else ''
                else:
                    internal_id_1 = codice_fondo
                    internal_id_1_type = 'Codice fondo' if codice_fondo else ''
                    internal_id_2 = ''
                    internal_id_2_type = ''

                sqldict['Name'].append(detail_name or name)
                sqldict['InternalID_1'].append(internal_id_1)
                sqldict['InternalID_1_type'].append(internal_id_1_type)
                sqldict['InternalID_2'].append(internal_id_2)
                sqldict['InternalID_2_type'].append(internal_id_2_type)
                # sqldict['CoType'].append(forma_giuridica)
                sqldict['Address_1'].append(address['Address_1'])
                sqldict['Address_2'].append(address['Address_2'])
                sqldict['City'].append(address['City'])
                sqldict['Zip'].append(address['Zip'])
                sqldict['Cntry'].append(address['Cntry'])
                sqldict['Website'].append(website)
                sqldict['RegulationDate'].append(data_iscrizione)

                append_common(sqldict, list_code, processdate, typology_name, regulation_type_from_status(stato))

    sqldict = bourange_same_length_array(sqldict)
    print(f"  Total rows so far: {len(sqldict['Name'])}")

[INFO] : Working 1/1 _(SM BCSM 7)_ 
  Fetching: https://www.bcsm.sm/registro-imprese-capogruppo
  Found 4 entity blocks
    Detail 4/4: BANCA SAMMARINESE DI INVESTIMENTO S.P.A.
  Total rows so far: 4


In [ ]:
#------------------------------------------------ Begin_writer and save df to excel ----------------------------------------
os.chdir(scriptfolder)
df = pd.DataFrame(sqldict)

df = df[df['RegulationType']=='Regulated']
df.to_excel(filename, sheet_name='SQL Ready', index=False)
print(f"[INFO] : Excel file '{filename}' saved successfully")

In [7]:
df

,bvdid,priority,ListLabel,Typology,EntryType,Name,InternalID_1,InternalID_1_type,InternalID_2,InternalID_2_type,...,LEI Code,BIC SWIFT Code,Name - Mother Company,Address_1 - Mother company,Address_2 - Mother company,City - Mother company,Zip - Mother company,Cntry - Mother company,Phone - Mother company,Check
0,,,,Impresa Capogruppo Finanziaria,,BANCA DI SAN MARINO S.P.A.,SM00476 Registro delle Imprese Capogruppo,Codice Operatore Economico,IC002,Numero iscrizione (Registro imprese capogruppo),...,,,,,,,,,,
1,,,,Impresa Capogruppo Finanziaria,,BANCA AGRICOLA COMMERCIALE ISTITUTO BANCARIO S...,SM00087 Registro delle Imprese Capogruppo,Codice Operatore Economico,IC004,Numero iscrizione (Registro imprese capogruppo),...,,,,,,,,,,
2,,,,Impresa Capogruppo Finanziaria,,CASSA DI RISPARMIO DELLA REPUBBLICA DI SAN MAR...,SM00099 Registro delle Imprese Capogruppo,Codice Operatore Economico,IC005,Numero iscrizione (Registro imprese capogruppo),...,,,,,,,,,,
3,,,,Impresa Capogruppo Finanziaria,,BANCA SAMMARINESE DI INVESTIMENTO S.P.A.,SM18493 Registro delle Imprese Capogruppo,Codice Operatore Economico,IC006,Numero iscrizione (Registro imprese capogruppo),...,,,,,,,,,,


In [8]:
#------------------------------------------------ Data Integrity & Consistency Check ----------------------------------------

print("=" * 80)
print("DATA INTEGRITY & CONSISTENCY VERIFICATION")
print("=" * 80)

expected_lists = {
    '1': {'name': 'Register of authorized entities'},
    '3': {'name': 'Register of insurance and reinsurance intermediaries'},
    '5': {'name': 'Register of payment service providers'},
    '6': {'name': 'Register of Professional Trustees in the Republic Of San Marino'},
    '7': {'name': 'Register of holding companies'},
    '8': {'name': 'List of Approved Funds (EFA)'},
}

print(f"\n1. DATAFRAME SHAPE:")
print(f"   Total rows collected: {len(df)}")
print(f"   Total columns: {len(df.columns)}")

print("\n2. DATA DISTRIBUTION BY LIST:")
if len(df) > 0:
    list_summary = df.groupby('ListCode').agg({
        'Name': 'count',
        'ListName': 'first',
        'RegCtry': 'first',
        'RegCode': 'first'
    }).rename(columns={'Name': 'Count'})
    print(list_summary)
else:
    print("   NO DATA COLLECTED")

print("\n3. NULL VALUES CHECK (Data Completeness):")
null_counts = df.isnull().sum()
if null_counts.sum() == 0:
    print("   PASS: No null values found")
else:
    print("   FAIL: Null values detected in:")
    for col, count in null_counts[null_counts > 0].items():
        print(f"      - {col}: {count} nulls")

print("\n4. CONSISTENCY WITH README:")
print("\n   List | Expected Name                                                       | Count | Status")
print("   -----|---------------------------------------------------------------------------|-------|--------")
for list_code, expected in expected_lists.items():
    data = df[df['ListCode'] == list_code]
    count = len(data)
    if count == 0:
        print(f"   {list_code:<4} | {expected['name']:<73} | {count:>5} | MISSING")
    else:
        print(f"   {list_code:<4} | {expected['name']:<73} | {count:>5} | OK")

print("\n5. REGCTRY & REGCODE VALIDATION:")
if len(df) > 0:
    regctry_values = df['RegCtry'].unique()
    regcode_values = df['RegCode'].unique()
    regctry_status = "CORRECT" if all(v == 'SM' for v in regctry_values) else "INCORRECT"
    regcode_status = "CORRECT" if all(v == 'BCSM' for v in regcode_values) else "INCORRECT"
    print(f"   RegCtry values: {regctry_values} {regctry_status}")
    print(f"   RegCode values: {regcode_values} {regcode_status}")
else:
    print("   NO DATA TO VALIDATE")

print("\n6. SAMPLE DATA (first 5 rows):")
if len(df) > 0:
    print(df[['Name', 'InternalID_1', 'RegulationType', 'ListCode', 'ListName']].head(5).to_string())
else:
    print("   NO DATA COLLECTED")

print("\n" + "=" * 80)
print("SUMMARY:")
print("=" * 80)
print(f"Total rows in DataFrame: {len(df)}")
total_expected = len(expected_lists)
total_collected = len(df['ListCode'].unique()) if len(df) > 0 else 0
print(f"Expected lists coverage: {total_collected}/{total_expected}")
print("=" * 80)

DATA INTEGRITY & CONSISTENCY VERIFICATION

1. DATAFRAME SHAPE:
   Total rows collected: 4
   Total columns: 44

2. DATA DISTRIBUTION BY LIST:
          Count                       ListName RegCtry RegCode
ListCode                                                      
7             4  Register of holding companies      SM    BCSM

3. NULL VALUES CHECK (Data Completeness):
   PASS: No null values found

4. CONSISTENCY WITH README:

   List | Expected Name                                                       | Count | Status
   -----|---------------------------------------------------------------------------|-------|--------
   1    | Register of authorized entities                                           |     0 | MISSING
   3    | Register of insurance and reinsurance intermediaries                      |     0 | MISSING
   5    | Register of payment service providers                                     |     0 | MISSING
   6    | Register of Professional Trustees in the Republic Of 